In [1]:
#relevant imports to prepare data
import json
import re
import glob
import os
from collections import defaultdict
import unicodedata


In [2]:
#path to filtered tax ruling json files (path leverages on volume mounting)
path_to_files = "/home/jovyan/data/processed/"

In [3]:
#Normalization functions to strip accents, capitalize characters. 
#Relevant for data preparation
def strip_accents(s: str) -> str:
    return "".join(
        c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c)
    )


def norm_tipo(s: str) -> str:
    return re.sub(r"\s+", " ", strip_accents(s or "")).strip().upper()


In [4]:
#Base Regex to identify the decision section of the ruling.
RE_DECISION_BLOCK = re.compile(
    r"DECISION\b.*?"
    r"ADMINISTRANDO\s+JUSTICIA[\s,]+(?:EN\s+)?NOMBRE\s+DEL\s+PUEBLO\s+SOBERANO\s+"
    r"DEL\s+ECUADOR[\s,]*(?:Y\s+)?POR\s+AUTORIDAD\s+DE\s+LA\s+CONSTITUCION\s+Y\s+"
    r"LAS\s+LEYES\s+DE\s+LA\s+REPUBLICA[\s,]*(.*)", 
    re.DOTALL
)
#Regex to match the three types of rulings from the taxpayer's perspective:
#Accept the judicial challenge, deny the judicial challenge or partially accept the judicial challenge. 
RE_PARTIAL = re.compile(
    r"ACEPT\w*\s+PARCIAL|PARCIALMENTE|ACCOGER?\s+PARCIAL"
)

RE_ACCEPT = re.compile(
    r"\b(ACEPT\w*|ACOG\w*)\s+(LA\s+|SU\s+)?DEMANDA"
    r"|\bSE\s+ACEPTA\b"
    r"|\bDECLARAR\s+(LA\s+)?VALIDEZ\b"  
)

RE_DENY = re.compile(
    r"\b(NIEG\w*|NEG\w*|RECHAZ\w*|DESECH\w*)\s+(LA\s+|SU\s+)?DEMANDA"
    r"|\bSIN\s+LUGAR\b"
    r"|\bIMPROCEDENTE\b"
    r"|\bDECLARAR\s+SIN\s+LUGAR\b"
)


In [5]:
#find all jsonfiles in directory
json_files = glob.glob(os.path.join(path_to_files, "*.json"))

In [6]:
#Matching, filtering and taggin section. 
#Here is the basis of the functions that will be used to process several tax rulings (data) and to tag it. 
#Use regex to match the relevant part of the ruling where a decision is made
#Use regex to identify the type of ruling and to register it inside a dictionary.
match_counter = 0
ruling_outcome = {"accept": 0, "partial": 0, "denied": 0, "no matches found": 0}
for file_path in json_files:
    with open(file_path, 'rb') as file:
        if file.read(2) != b'[]':
            file_name = os.path.basename(file_path)
            with open(file_path, "r", encoding="utf-8") as f:
                try:
                    data = json.load(f)
                    for item in data:
                        ruling = item.get("actividad")
                        relevant_ruling = ruling[-2500:]
                        clean_ruling = norm_tipo(relevant_ruling)
                        #print(clean_ruling)
                        match = RE_DECISION_BLOCK.search(clean_ruling)
                        if match:
                            match_counter += 1
                            # .strip() cleans up any leading or trailing whitespaces/newlines
                            d = match.group(1).strip()
                            d = norm_tipo(d)
                            #print(d)
                            if RE_PARTIAL.search(d):
                                ruling_outcome["partial"] += 1
                                #print(f"partial {d}")
                            elif RE_ACCEPT.search(d):
                                ruling_outcome["accept"] += 1
                                #print(f"accept {d}")
                            elif RE_DENY.search(d):
                                ruling_outcome["denied"] += 1
                               #print(f"denied {d}")
                            else:
                                ruling_outcome["no matches found"] += 1
                        else:
                            print("No match found.")
                except json.JSONDecodeError:
                    print(f"Error: {file_name} is not a valid JSON file.")

In [7]:
print(ruling_outcome)

{'accept': 1, 'partial': 1, 'denied': 1, 'no matches found': 0}


In [8]:
print(match_counter)

3
